In [ ]:
from __future__ import annotations
import os
import argparse
from typing import Tuple, List

import cv2
import numpy as np
import pandas as pd
from scipy.ndimage import label


def dark_ratio(img: np.ndarray, dark_thresh: int = 40) -> float:
    """Return fraction of pixels darker than *dark_thresh* (0‑255)."""
    return float((img < dark_thresh).sum()) / img.size


def image_dark_percent(image_path: str, dark_thresh: int = 40) -> float:
    """Decode image from disk and return percentage of dark pixels (0‑1)."""
    img = cv2.imdecode(np.fromfile(image_path, dtype=np.uint8),
                       cv2.IMREAD_GRAYSCALE)
    return dark_ratio(img, dark_thresh)


def analyse_folder_darkness(folder: str,
                            dark_thresh: int = 40,
                            csv_path: str | None = None) -> pd.DataFrame:
    rows = []
    for fname in os.listdir(folder):
        if not fname.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
            continue
        pct = image_dark_percent(os.path.join(folder, fname), dark_thresh) * 100
        rows.append({"image": fname, "dark_%": pct})
    df = pd.DataFrame(rows).sort_values("dark_%", ascending=False)
    if csv_path:
        df.to_csv(csv_path, index=False)
    return df



In [ ]:
def preprocess(img: np.ndarray,
               brightness: float = 1.3,
               clahe_clip: float = 2.0,
               denoise_h: int = 20) -> np.ndarray:
    img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX)
    img = cv2.convertScaleAbs(img, alpha=brightness, beta=0)
    clahe = cv2.createCLAHE(clipLimit=clahe_clip, tileGridSize=(8, 8))
    img = clahe.apply(img)
    img = cv2.fastNlMeansDenoising(img, None, h=denoise_h,
                                   templateWindowSize=7, searchWindowSize=21)
    return img


In [ ]:

def tune_params(tile: np.ndarray,
                low: float = 0.40,
                mid: float = 0.20,
                area_min_urban: float = 0.001) -> Tuple[float, float]:
    dr = dark_ratio(tile)
    if dr > low:       # water‑heavy (mostly dark)
        return area_min_urban / 5, 1.5
    if dr > mid:       # mixed
        return area_min_urban / 2, 1.8
    return area_min_urban, 2.2  # urban / bright

In [ ]:

def detect_objects(tile: np.ndarray,
                   area_thresh: float,
                   intensity_ratio: float) -> List[Tuple[int, int, int, int]]:
    pre = preprocess(tile)
    bin_img = cv2.adaptiveThreshold(pre, 255,
                                    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                    cv2.THRESH_BINARY, 51, -5)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    bin_img = cv2.morphologyEx(bin_img, cv2.MORPH_OPEN, k, iterations=1)
    bin_img = cv2.morphologyEx(bin_img, cv2.MORPH_CLOSE, k, iterations=2)

    labeled, num = label(bin_img)
    objs = []
    min_pixels = int(area_thresh * tile.shape[0] * tile.shape[1])
    bg_med = np.median(tile)

    for i in range(1, num + 1):
        ys, xs = np.where(labeled == i)
        if xs.size < min_pixels:
            continue
        if tile[ys, xs].mean() < intensity_ratio * bg_med:
            continue
        objs.append((xs.min(), ys.min(), xs.max(), ys.max()))
    return objs

In [ ]:

def split_image(image: np.ndarray, max_tile_size=(2000, 2000)):
    h, w = image.shape
    if h <= max_tile_size[1] and w <= max_tile_size[0]:
        return [(image, 0, 0)]
    tw, th = max_tile_size
    tiles = []
    for y in range(0, h, th):
        for x in range(0, w, tw):
            tiles.append((image[y:y + th, x:x + tw], x, y))
    return tiles

In [ ]:
def process_image(image_path: str,
                  max_tile_size=(2000, 2000),
                  low: float = 0.40,
                  mid: float = 0.20,
                  area_min: float = 0.001):
    img = cv2.imdecode(np.fromfile(image_path, dtype=np.uint8),
                       cv2.IMREAD_GRAYSCALE)
    boxes = []
    for tile, xo, yo in split_image(img, max_tile_size):
        area_t, ratio_t = tune_params(tile, low, mid, area_min)
        for x1, y1, x2, y2 in detect_objects(tile, area_t, ratio_t):
            boxes.append((x1 + xo, y1 + yo, x2 + xo, y2 + yo))
    return boxes


def draw_objects(image_path: str, objects, output_path: str):
    img = cv2.imdecode(np.fromfile(image_path, dtype=np.uint8),
                       cv2.IMREAD_COLOR)
    for x1, y1, x2, y2 in objects:
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
    cv2.imwrite(output_path, img)


def process_folder(input_folder: str,
                   output_folder: str,
                   max_tile_size=(2000, 2000),
                   low: float = 0.40,
                   mid: float = 0.20,
                   area_min: float = 0.001):
    os.makedirs(output_folder, exist_ok=True)
    summary = []
    for fname in os.listdir(input_folder):
        if not fname.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
            continue
        inp = os.path.join(input_folder, fname)

        # ---- darkness analysis ----
        pct_dark = image_dark_percent(inp) * 100
        if pct_dark > low * 100:
            scene = "water‑heavy"
        elif pct_dark > mid * 100:
            scene = "mixed"
        else:
            scene = "urban"
        print(f"{fname}: dark {pct_dark:.1f}% → {scene} params")

        # ---- detection ----
        boxes = process_image(inp, max_tile_size, low, mid, area_min)
        out = os.path.join(output_folder, f"{os.path.splitext(fname)[0]}_out.jpg")
        draw_objects(inp, boxes, out)
        summary.append({"image": fname,
                        "dark_%": pct_dark,
                        "objects": len(boxes),
                        "scene": scene,
                        "output": out})

    if summary:
        pd.DataFrame(summary).to_csv(os.path.join(output_folder, "summary.csv"),
                                     index=False)
        print("CSV summary saved → summary.csv")

In [ ]:
if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Context‑aware SAR detector")
    parser.add_argument("--input", default="./images", help="Input folder with SAR images")
    parser.add_argument("--output", default="./out", help="Output folder for results")
    parser.add_argument("--tile", type=int, default=2000, help="Max tile size (pixels)")
    parser.add_argument("--low", type=float, default=0.40, help="Dark‑pixel ratio split for water‑heavy scene")
    parser.add_argument("--mid", type=float, default=0.20, help="Dark‑pixel ratio split for mixed scene")
    parser.add_argument("--area_min", type=float, default=0.001, help="Min blob area fraction for urban tiles")
    parser.add_argument("--brightness", type=float, default=1.3, help="Global brightness gain (alpha)")
    parser.add_argument("--denoise", type=int, default=20, help="fastNlMeans strength (higher = smoother)")
    args, _ = parser.parse_known_args()  # ignore extra args (e.g., Jupyter "-f")

    # patch preprocess defaults according to CLI knobs
    preprocess.__defaults__ = (args.brightness, 2.0, args.denoise)

    process_folder(args.input, args.output,
                   max_tile_size=(args.tile, args.tile),
                   low=args.low, mid=args.mid, area_min=args.area_min)